# 01 · Casting Product - Exploratory Data Analysis

**Arkon Manufacturing | Department: Foundry**

Dataset: binary classification of casting defects (def_front / ok_front)
- Train: `train/def_front/` + `train/ok_front/`
- Test: `test/def_front/` + `test/ok_front/`

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter
import os

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
ASSETS = 'cv/casting'


In [ ]:
DATA_DIR = Path('../../../data/03_casting/raw')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

assert DATA_DIR.exists(), f'Data not found: {DATA_DIR}'
print(f'Train classes: {[d.name for d in TRAIN_DIR.iterdir() if d.is_dir()]}')
print(f'Test  classes: {[d.name for d in TEST_DIR.iterdir() if d.is_dir()]}')

## 1. Dataset Overview - image count per class

In [ ]:
def count_images(root: Path) -> dict:
    result = {}
    for cls_dir in sorted(root.iterdir()):
        if cls_dir.is_dir():
            imgs = list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
            result[cls_dir.name] = len(imgs)
    return result

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

print('TRAIN:', train_counts)
print('TEST: ', test_counts)
print(f'Total: {sum(train_counts.values()) + sum(test_counts.values())} images')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (split, counts) in zip(axes, [('Train', train_counts), ('Test', test_counts)]):
    ax.bar(counts.keys(), counts.values(), color=['#e74c3c', '#2ecc71'])
    ax.set_title(f'{split} set')
    ax.set_ylabel('Images')
    for i, v in enumerate(counts.values()):
        ax.text(i, v + 5, str(v), ha='center')
plt.suptitle('Casting Product - Class Distribution')
plt.tight_layout()
save_figure(fig, 'casting_eda_class_distribution', subfolder=ASSETS)
plt.show()

## 3. Sample Images - examples from each class

In [ ]:
def show_samples(root: Path, n: int = 5):
    classes = sorted([d for d in root.iterdir() if d.is_dir()])
    fig, axes = plt.subplots(len(classes), n, figsize=(15, 4))
    for row, cls_dir in enumerate(classes):
        imgs = list(cls_dir.glob('*.jpeg'))[:n]
        for col, img_path in enumerate(imgs):
            img = Image.open(img_path)
            axes[row][col].imshow(img, cmap='gray')
            axes[row][col].axis('off')
            if col == 0:
                axes[row][col].set_ylabel(cls_dir.name, fontsize=12)
    plt.suptitle('Sample Images (Train)', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'casting_eda_class_distribution', subfolder=ASSETS)
plt.show()

show_samples(TRAIN_DIR)

## 4. Image Statistics - sizes and channels

In [ ]:
def get_image_stats(root: Path, sample_size: int = 50) -> list:
    stats = []
    for cls_dir in root.iterdir():
        if cls_dir.is_dir():
            for img_path in list(cls_dir.glob('*.jpeg'))[:sample_size]:
                img = Image.open(img_path)
                stats.append({'class': cls_dir.name, 'width': img.size[0],
                               'height': img.size[1], 'mode': img.mode})
    return stats

import pandas as pd
df_stats = pd.DataFrame(get_image_stats(TRAIN_DIR))
print(df_stats.groupby('class')[['width', 'height']].describe().round(1))
print(f"\nColor modes: {df_stats['mode'].value_counts().to_dict()}")

## Summary

| Parameter | Value |
|---|---|
| Task | Binary classification |
| Classes | def_front (defect) / ok_front (normal) |
| Framework | PyTorch + torchvision |
| Model | ResNet18 (Transfer Learning) |

➡️ **Next step:** `02_casting_preprocessing.ipynb`